# Exp9 — WTE for HalfCheetahVel task distributions

Same pipeline as `exp2.ipynb` (collect SARS → LWE/WTE embeddings → within/between cosine distances + PCA/LDA),
but for the three CheetahVel velocity ranges. Run from `scripts/` or the repo root with the `adv_rl` conda env.

| ID | Range |
|----|--------|
| 1 | Uniform\[0.5, 0.75\] |
| 2 | Uniform\[0.75, 1.0\] |
| 3 | Uniform\[2.5, 2.75\] |
| 4 | Uniform\[2.75, 3.0\] |
| 5 | Uniform\[0.0, 3.0\] |

Continuous actions → `Detect(..., one_hot=False)`. Discrete tasks within a dist are sampled velocities (linspace).

Data is written under `data/chevel/task_dist{1..5}/`. Eval returns from `run_chevel.sh` are the CSV files in `data/chevel/` (plotted at the end of this notebook).

In [1]:
%matplotlib inline

import sys
from pathlib import Path

from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Repo root when launched from scripts/ or from the repo root.
CODE_DIR = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
for p in (CODE_DIR, CODE_DIR / "scripts", CODE_DIR / "hyper"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import environments  # registers HalfCheetahVel-TaskDist*-v0
from environments.mujoco.half_cheetah_vel import HalfCheetahVelEnv
from detect import Detect

DATA_ROOT = CODE_DIR / "data" / "chevel"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print("CODE_DIR", CODE_DIR)
print("DATA_ROOT", DATA_ROOT)

CODE_DIR /home/mahesh/hypex
DATA_ROOT /home/mahesh/hypex/data/chevel


/home/mahesh/miniconda3/envs/adv_rl/lib/python3.10/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647352509/work/c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
# CheetahVel task distributions (equal-width narrow bands + full range)
VEL_TASK_DISTS = {
    1: (0.5, 0.75),   # low band A
    2: (0.75, 1.0),   # low band B
    3: (2.5, 2.75),   # high band A
    4: (2.75, 3.0),   # high band B
    5: (0.0, 3.0),    # full range
}
N_TASKS_PER_DIST = 8  # discrete velocities sampled inside each Uniform range


def sample_velocities(dist_id, n_tasks=None):
    """Discrete velocity tasks drawn from Uniform[low, high] via linspace (inclusive)."""
    if n_tasks is None:
        n_tasks = N_TASKS_PER_DIST
    lo, hi = VEL_TASK_DISTS[dist_id]
    return [float(v) for v in np.linspace(lo, hi, n_tasks)]

In [3]:
EP_LEN = 200
N_EPISODES = 25          # rollouts per velocity task
N = EP_LEN * N_EPISODES   # SARS rows kept per velocity (~5000, like exp2)
SEED = 42

OBS_DIM = 20   # HalfCheetahVel hyper obs
ACTION_DIM = 6  # continuous torque dim (not one-hot)


def vel_filename(vel):
    return f"vel_{vel:.4f}.npy"


def collect_tuples(task_dists, seed=SEED):
    """task_dists: {dist_id: list of target velocities}.

    Saves SARS arrays under data/chevel/task_dist{id}/vel_X.XXXX.npy
    Each row: [s (20), a (6), r (1), s' (20)] → shape (N, 47).
    """
    rng = np.random.default_rng(seed)
    for dist_id, velocities in task_dists.items():
        lo, hi = VEL_TASK_DISTS[dist_id]
        out_dir = DATA_ROOT / f"task_dist{dist_id}"
        out_dir.mkdir(parents=True, exist_ok=True)

        env = HalfCheetahVelEnv(max_episode_steps=EP_LEN, vel_low=lo, vel_high=hi)
        for vel in velocities:
            sar_rows = []
            env.reset_task(vel)
            obs, _ = env.reset(seed=int(rng.integers(0, 1_000_000)))

            while len(sar_rows) < N:
                action = rng.uniform(
                    env.action_space.low, env.action_space.high
                ).astype(np.float64)
                next_obs, reward, terminated, truncated, info = env.step(action)
                row = np.concatenate(
                    [np.asarray(obs, dtype=np.float64).ravel(),
                     action.ravel(),
                     np.asarray([reward], dtype=np.float64),
                     np.asarray(next_obs, dtype=np.float64).ravel()]
                )
                sar_rows.append(row)
                obs = next_obs
                if terminated or truncated:
                    env.reset_task(vel)
                    obs, _ = env.reset()

            sar = np.asarray(sar_rows[:N], dtype=np.float64)
            path = out_dir / vel_filename(vel)
            np.save(path, sar)
            print(f"dist {dist_id} vel={vel:.4f} → {path.name} shape={sar.shape}")
        env.close()


def mean_pairwise_cosine_distance(embeddings):
    """Mean pairwise cosine distance (1 - cos) after L2-normalizing each embedding.

    Already an average over pairs (not a sum), so more tasks do not mechanically
    inflate the score — but unequal task counts still make cross-env comparison
    imperfect; use mean_pairwise_matched(..., PAIRWISE_SUBSET_SIZE) for that.
    """
    E = np.stack([np.asarray(e).ravel() for e in embeddings], axis=0)
    norms = np.linalg.norm(E, axis=1, keepdims=True)
    E = E / np.clip(norms, 1e-12, None)
    cos_dists = [
        1.0 - float(np.dot(E[i], E[j]))
        for i in range(len(E))
        for j in range(i + 1, len(E))
    ]
    return float(np.mean(cos_dists)) if cos_dists else float("nan")


def evenly_spaced_indices(n, k):
    """k indices in [0, n) including endpoints (same coverage idea as linspace)."""
    if k >= n:
        return list(range(n))
    return sorted({int(round(i)) for i in np.linspace(0, n - 1, k)})


def mean_pairwise_matched(embeddings, subset_size):
    """Fair mean pairwise when TDs have different #tasks.

    If n <= subset_size: ordinary mean pairwise.
    If n > subset_size: average of mean-pairwise over *all* subsets of size
    subset_size (U-statistic). Use the same subset_size in exp2 and exp9
    (default 4 = #goals per GridNavi TD).
    """
    embs = [np.asarray(e).ravel() for e in embeddings]
    n = len(embs)
    if n <= subset_size:
        return mean_pairwise_cosine_distance(embs)
    return float(np.mean([
        mean_pairwise_cosine_distance(list(subset))
        for subset in combinations(embs, subset_size)
    ]))


def mean_pairwise_wte(dist_id, velocities):
    """Mean pairwise cosine distance of WTE/LWE embeddings within one task dist."""
    out_dir = DATA_ROOT / f"task_dist{dist_id}"
    embeddings = []
    for vel in velocities:
        sar = np.load(out_dir / vel_filename(vel))
        embeddings.append(np.asarray(detect.lwe(sar, ACTION_DIM)).ravel())
    return mean_pairwise_cosine_distance(embeddings)

In [4]:
# Sample discrete velocities inside each Uniform range and collect SARS
task_dists = {dist_id: sample_velocities(dist_id) for dist_id in VEL_TASK_DISTS}
for dist_id, vels in task_dists.items():
    lo, hi = VEL_TASK_DISTS[dist_id]
    print(f"TaskDist{dist_id} Uniform[{lo}, {hi}] → {len(vels)} vels: {[round(v, 3) for v in vels]}")

collect_tuples(task_dists)

TaskDist1 Uniform[0.5, 0.75] → 8 vels: [0.5, 0.536, 0.571, 0.607, 0.643, 0.679, 0.714, 0.75]
TaskDist2 Uniform[0.75, 1.0] → 8 vels: [0.75, 0.786, 0.821, 0.857, 0.893, 0.929, 0.964, 1.0]
TaskDist3 Uniform[2.5, 2.75] → 8 vels: [2.5, 2.536, 2.571, 2.607, 2.643, 2.679, 2.714, 2.75]
TaskDist4 Uniform[2.75, 3.0] → 8 vels: [2.75, 2.786, 2.821, 2.857, 2.893, 2.929, 2.964, 3.0]
TaskDist5 Uniform[0.0, 3.0] → 8 vels: [0.0, 0.429, 0.857, 1.286, 1.714, 2.143, 2.571, 3.0]


dist 1 vel=0.5000 → vel_0.5000.npy shape=(5000, 47)
dist 1 vel=0.5357 → vel_0.5357.npy shape=(5000, 47)
dist 1 vel=0.5714 → vel_0.5714.npy shape=(5000, 47)
dist 1 vel=0.6071 → vel_0.6071.npy shape=(5000, 47)
dist 1 vel=0.6429 → vel_0.6429.npy shape=(5000, 47)
dist 1 vel=0.6786 → vel_0.6786.npy shape=(5000, 47)
dist 1 vel=0.7143 → vel_0.7143.npy shape=(5000, 47)
dist 1 vel=0.7500 → vel_0.7500.npy shape=(5000, 47)


dist 2 vel=0.7500 → vel_0.7500.npy shape=(5000, 47)
dist 2 vel=0.7857 → vel_0.7857.npy shape=(5000, 47)
dist 2 vel=0.8214 → vel_0.8214.npy shape=(5000, 47)
dist 2 vel=0.8571 → vel_0.8571.npy shape=(5000, 47)
dist 2 vel=0.8929 → vel_0.8929.npy shape=(5000, 47)
dist 2 vel=0.9286 → vel_0.9286.npy shape=(5000, 47)
dist 2 vel=0.9643 → vel_0.9643.npy shape=(5000, 47)
dist 2 vel=1.0000 → vel_1.0000.npy shape=(5000, 47)


dist 3 vel=2.5000 → vel_2.5000.npy shape=(5000, 47)
dist 3 vel=2.5357 → vel_2.5357.npy shape=(5000, 47)
dist 3 vel=2.5714 → vel_2.5714.npy shape=(5000, 47)
dist 3 vel=2.6071 → vel_2.6071.npy shape=(5000, 47)
dist 3 vel=2.6429 → vel_2.6429.npy shape=(5000, 47)
dist 3 vel=2.6786 → vel_2.6786.npy shape=(5000, 47)
dist 3 vel=2.7143 → vel_2.7143.npy shape=(5000, 47)
dist 3 vel=2.7500 → vel_2.7500.npy shape=(5000, 47)


dist 4 vel=2.7500 → vel_2.7500.npy shape=(5000, 47)
dist 4 vel=2.7857 → vel_2.7857.npy shape=(5000, 47)
dist 4 vel=2.8214 → vel_2.8214.npy shape=(5000, 47)
dist 4 vel=2.8571 → vel_2.8571.npy shape=(5000, 47)
dist 4 vel=2.8929 → vel_2.8929.npy shape=(5000, 47)
dist 4 vel=2.9286 → vel_2.9286.npy shape=(5000, 47)
dist 4 vel=2.9643 → vel_2.9643.npy shape=(5000, 47)
dist 4 vel=3.0000 → vel_3.0000.npy shape=(5000, 47)


dist 5 vel=0.0000 → vel_0.0000.npy shape=(5000, 47)
dist 5 vel=0.4286 → vel_0.4286.npy shape=(5000, 47)
dist 5 vel=0.8571 → vel_0.8571.npy shape=(5000, 47)
dist 5 vel=1.2857 → vel_1.2857.npy shape=(5000, 47)
dist 5 vel=1.7143 → vel_1.7143.npy shape=(5000, 47)
dist 5 vel=2.1429 → vel_2.1429.npy shape=(5000, 47)
dist 5 vel=2.5714 → vel_2.5714.npy shape=(5000, 47)
dist 5 vel=3.0000 → vel_3.0000.npy shape=(5000, 47)


In [ ]:
# Visualise velocity spreads + why same absolute width ≠ same WTE diversity.
# Absolute band width is 0.25 for TD1–4, but mid-velocity and Δv/mid differ,
# and random-action SARS (rewards / tracking error) also change with |target|.

rows_spread = []
for dist_id, (lo, hi) in sorted(VEL_TASK_DISTS.items()):
    vels = np.asarray(task_dists[dist_id], dtype=float)
    width = hi - lo
    mid = 0.5 * (lo + hi)
    spacing = float(np.diff(vels).mean()) if len(vels) > 1 else 0.0
    rows_spread.append({
        "dist": dist_id,
        "lo": lo,
        "hi": hi,
        "width": width,
        "mid": mid,
        "rel_width": width / mid if mid > 0 else float("nan"),  # Δv / v_mid
        "linspace_spacing": spacing,
        "n_tasks": len(vels),
    })
df_spread = pd.DataFrame(rows_spread)
print("Band geometry (TD1–4 share absolute width 0.25; relative width differs):")
display(df_spread.round(4))

# --- panel A: ranges + sampled target velocities ---
fig, axes = plt.subplots(3, 1, figsize=(10, 9), gridspec_kw={"height_ratios": [1.1, 1.0, 1.2]})

ax = axes[0]
for i, (dist_id, (lo, hi)) in enumerate(sorted(VEL_TASK_DISTS.items())):
    y = i
    ax.hlines(y, lo, hi, colors=f"C{i}", linewidth=8, alpha=0.35)
    vels = task_dists[dist_id]
    ax.scatter(vels, [y] * len(vels), c=f"C{i}", s=45, zorder=3, label=f"TD{dist_id}")
    for v in vels:
        ax.axvline(v, color=f"C{i}", alpha=0.08, linewidth=0.8)
ax.set_yticks(range(len(VEL_TASK_DISTS)))
ax.set_yticklabels([f"TD{d}  [{lo},{hi}]" for d, (lo, hi) in sorted(VEL_TASK_DISTS.items())])
ax.set_xlabel("target velocity")
ax.set_xlim(-0.1, 3.2)
ax.set_title("A — Sampled target velocities (linspace inside each Uniform band)")
ax.legend(loc="upper right", fontsize=8, ncol=5)

# --- panel B: absolute vs relative width ---
ax = axes[1]
narrow = df_spread[df_spread["dist"] != 5]
x = np.arange(len(narrow))
w = 0.35
ax.bar(x - w / 2, narrow["width"], width=w, label="absolute width (hi−lo)", color="C0", alpha=0.8)
ax.bar(x + w / 2, narrow["rel_width"], width=w, label="relative width (hi−lo)/mid", color="C1", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f"TD{int(d)}" for d in narrow["dist"]])
ax.set_ylabel("width")
ax.set_title("B — Same absolute width, different relative width (Δv / v_mid)")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)

# --- panel C: reward stats from collected SARS (random actions) ---
# SARS row: [s(20), a(6), r(1), s'(20)]; goal velocity is last dim of s (index 19).
ax = axes[2]
rew_rows = []
for dist_id, vels in sorted(task_dists.items()):
    out_dir = DATA_ROOT / f"task_dist{dist_id}"
    for vel in vels:
        sar = np.load(out_dir / vel_filename(vel))
        r = sar[:, OBS_DIM + ACTION_DIM]  # reward column in [s,a,r,s']
        rew_rows.append({
            "dist": dist_id,
            "target_vel": vel,
            "mean_r": float(r.mean()),
            "std_r": float(r.std()),
            "mean_|r|": float(np.abs(r).mean()),
        })
df_rew = pd.DataFrame(rew_rows)
print("Per-task reward stats under random actions (from collected SARS):")
display(df_rew.groupby("dist")[["mean_r", "std_r", "mean_|r|"]].mean().round(4))

for dist_id in sorted(task_dists):
    sub = df_rew[df_rew["dist"] == dist_id]
    ax.plot(sub["target_vel"], sub["mean_r"], "o-", label=f"TD{dist_id}", color=f"C{dist_id-1}")
ax.axhline(0, color="k", linewidth=0.6, alpha=0.4)
ax.set_xlabel("target velocity")
ax.set_ylabel("mean reward (random policy)")
ax.set_title("C — Mean episode reward vs target vel (physics / tracking hardness changes with |v|)")
ax.legend(fontsize=8, ncol=5)
ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

# Highlight the TD1≈TD3 / TD2≈TD4 pairing in geometry + rewards
pairs = [(1, 3), (2, 4)]
print("Paired TDs (same absolute width 0.25; compare mid / rel_width / mean reward):")
for a, b in pairs:
    sa, sb = df_spread.set_index("dist").loc[a], df_spread.set_index("dist").loc[b]
    ra = df_rew[df_rew["dist"] == a]["mean_r"].mean()
    rb = df_rew[df_rew["dist"] == b]["mean_r"].mean()
    print(
        f"  TD{a} vs TD{b}: mid={sa['mid']:.3f} vs {sb['mid']:.3f}, "
        f"rel_width={sa['rel_width']:.3f} vs {sb['rel_width']:.3f}, "
        f"mean_r={ra:.3f} vs {rb:.3f}"
    )

In [ ]:
# Detect / LWE setup
# Protocol matches exp2 (GridNavi): Detect.lwe on SARS, REFERENCE_NUM=100,
# L2-normalize embeddings, then 1-cos pairwise. Continuous actions → one_hot=False.
REFERENCE_NUM = 100
VIZ = False

# Shared with exp2 for cross-env comparison:
PCA_DIM = 8                 # PCA target dim
PAIRWISE_SUBSET_SIZE = 4    # match #goals per GridNavi TD (cheetah has 8)

detect = Detect(
    reference_num=REFERENCE_NUM,
    input_dim=OBS_DIM,
    action_dim=ACTION_DIM,
    num_samples=N,
    sars_flag=True,
    one_hot=False,  # continuous CheetahVel actions
)
detect.set_reference(OBS_DIM, REFERENCE_NUM, ACTION_DIM)
print("reference shape", tuple(detect.ref.shape), "(expect N_ref × (20+6+1+20)=47)")
print(f"PCA_DIM={PCA_DIM} (set the same value in exp2 for cross-env comparison)")

In [ ]:
# Compute each task's WTE/LWE once (same Detect.lwe path as exp2), cache by TD.
# Then: between-TD cosine on mean embeddings (raw WTE).

embeddings_by_dist = {}  # dist_id -> list of 1d embeddings
rows, labels, vels = [], [], []

for dist_id, velocities in task_dists.items():
    out_dir = DATA_ROOT / f"task_dist{dist_id}"
    embeddings = []
    for vel in velocities:
        sar = np.load(out_dir / vel_filename(vel))
        embedding = np.asarray(detect.lwe(sar, ACTION_DIM)).ravel()
        embeddings.append(embedding)
        rows.append(embedding)
        labels.append(dist_id)
        vels.append(vel)
        if VIZ:
            print(f"--- dist {dist_id} vel={vel:.4f} sar={sar.shape} emb[:5]={embedding[:5]} ---")
    embeddings_by_dist[dist_id] = embeddings

X = np.stack(rows)
labels = np.asarray(labels)
print(f"cached {len(rows)} task embeddings, raw dim={X.shape[1]}")

mean_embeddings = [np.mean(np.stack(embeddings_by_dist[d]), axis=0) for d in embeddings_by_dist]
dist_ids = list(embeddings_by_dist.keys())
V = np.stack(mean_embeddings)
V = V / np.clip(np.linalg.norm(V, axis=1, keepdims=True), 1e-12, None)
D = 1.0 - (V @ V.T)
df_mean = (
    pd.DataFrame(D, index=dist_ids, columns=dist_ids)
    .rename_axis("dist")
    .reset_index()
)
print("cosine distances between mean embeddings (raw WTE):")
display(df_mean.round(4))

In [ ]:
# Within-TD mean pairwise on raw WTE.
# d_all: all pairs (8 vels).  d_matched: fair vs grid — average mean-pairwise
# over all 4-subsets (PAIRWISE_SUBSET_SIZE), matching #goals per GridNavi TD.
rows_within = []
for dist_id, embs in embeddings_by_dist.items():
    rows_within.append({
        "dist": dist_id,
        "n_tasks": len(embs),
        "d_all": mean_pairwise_cosine_distance(embs),
        "d_matched": mean_pairwise_matched(embs, PAIRWISE_SUBSET_SIZE),
    })
df_within = pd.DataFrame(rows_within)
print(f"raw WTE within-TD pairwise (matched subset_size={PAIRWISE_SUBSET_SIZE}):")
display(df_within.round(4))

In [ ]:
# Fair PCA comparison vs exp2:
# 1) Keep exactly PAIRWISE_SUBSET_SIZE tasks per TD (evenly spaced) so both
#    envs contribute the same #points/TD to the PCA fit and pairwise stats.
# 2) Fit PCA_DIM on that matched corpus, then within-TD mean pairwise on Z.
# Full-data PCA (all 8 vels/TD) is still used for the scatter plot below.

fair_rows, fair_labels, fair_vels = [], [], []
for dist_id, embs in embeddings_by_dist.items():
    idxs = evenly_spaced_indices(len(embs), PAIRWISE_SUBSET_SIZE)
    v_list = list(task_dists[dist_id])
    for i in idxs:
        fair_rows.append(embs[i])
        fair_labels.append(dist_id)
        fair_vels.append(v_list[i])

X_fair = np.stack(fair_rows)
labels_fair = np.asarray(fair_labels)
X_fair_n = X_fair / np.clip(np.linalg.norm(X_fair, axis=1, keepdims=True), 1e-12, None)

pca_dim_eff = min(X_fair.shape[0] - 1, X_fair.shape[1], PCA_DIM)
if pca_dim_eff < PCA_DIM:
    print(f"WARNING: PCA_DIM={PCA_DIM} > max feasible {pca_dim_eff}; using {pca_dim_eff}")

pca_fair = PCA(n_components=pca_dim_eff, random_state=0)
Z_fair = pca_fair.fit_transform(X_fair_n)
print(
    f"fair PCA corpus: {len(fair_rows)} tasks "
    f"({PAIRWISE_SUBSET_SIZE}/TD)  {X_fair.shape[1]} → {pca_dim_eff}  "
    f"var_sum={pca_fair.explained_variance_ratio_.sum():.4f}"
)

z_fair_by_dist = {d: [] for d in embeddings_by_dist}
for z, d in zip(Z_fair, labels_fair):
    z_fair_by_dist[int(d)].append(z)

df_within_pca = pd.DataFrame({
    "dist": list(z_fair_by_dist.keys()),
    "n_tasks": [len(v) for v in z_fair_by_dist.values()],
    "d_matched": [mean_pairwise_cosine_distance(v) for v in z_fair_by_dist.values()],
    "pca_dim": pca_dim_eff,
})
print(f"PCA within-TD pairwise (fair: {PAIRWISE_SUBSET_SIZE} tasks/TD, dim={pca_dim_eff}):")
display(df_within_pca.round(4))

# Scatter: PCA on *all* tasks (viz only; not used for the fair d table above)
X_n = X / np.clip(np.linalg.norm(X, axis=1, keepdims=True), 1e-12, None)
pca_viz = PCA(n_components=2, random_state=0)
Z = pca_viz.fit_transform(X_n)

fig, ax = plt.subplots(figsize=(10, 8))
for dist_id in sorted(set(labels.tolist())):
    mask = labels == dist_id
    sc = ax.scatter(Z[mask, 0], Z[mask, 1], s=60, label=f"dist {dist_id}")
    color = sc.get_facecolors()[0]
    ax.scatter(
        Z[mask, 0].mean(), Z[mask, 1].mean(),
        s=160, marker="x", linewidths=2, c=[color],
    )
for (x, y), v, d in zip(Z, vels, labels):
    ax.annotate(f"{d}:{v:.2f}", (x, y), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_title("PCA — CheetahVel WTE")
ax.legend(fontsize=8)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
fig.tight_layout()
plt.show()

In [ ]:
# LDA of per-velocity task embeddings
n_classes = len(np.unique(labels))
n_components = min(2, n_classes - 1)
lda = LinearDiscriminantAnalysis(
    n_components=n_components, solver="eigen", shrinkage="auto"
)
Z_lda = lda.fit_transform(X_n, labels)

fig, ax = plt.subplots(figsize=(10, 8))
for dist_id in sorted(set(labels.tolist())):
    mask = labels == dist_id
    if Z_lda.shape[1] == 1:
        xs = Z_lda[mask, 0]
        ys = np.zeros_like(xs) + 0.02 * (dist_id - 2)
        sc = ax.scatter(xs, ys, s=60, label=f"dist {dist_id}")
        color = sc.get_facecolors()[0]
        ax.scatter(xs.mean(), ys.mean(), s=160, marker="x", linewidths=2, c=[color])
    else:
        sc = ax.scatter(Z_lda[mask, 0], Z_lda[mask, 1], s=60, label=f"dist {dist_id}")
        color = sc.get_facecolors()[0]
        ax.scatter(
            Z_lda[mask, 0].mean(), Z_lda[mask, 1].mean(),
            s=160, marker="x", linewidths=2, c=[color],
        )
ax.set_title("LDA — CheetahVel WTE embeddings")
ax.legend(fontsize=8)
ax.set_xlabel("LD1")
ax.set_ylabel("LD2" if Z_lda.shape[1] > 1 else "")
fig.tight_layout()
plt.show()

## `run_chevel.sh` eval returns (RNN+HN ablations on TaskDist3)

CSV exports of TensorBoard **Meta-Episode Return** live under `data/chevel/`:
`logs_HalfCheetahVel-TaskDist3-v0_chevel_td3_rnn_{full,head,film}_*.csv`

(`Step` = env frames; `Value` = eval return.)

In [ ]:
# Load run_chevel Meta-Episode Return CSVs
CHE_VEL_RETURN_GLOB = "logs_HalfCheetahVel-TaskDist3-v0_chevel_td3_rnn_*.csv"
METHOD_ORDER = ("full", "head", "film")
METHOD_COLORS = {"full": "C0", "head": "C1", "film": "C2"}


def parse_chevel_method(path: Path) -> str:
    name = path.name
    for m in METHOD_ORDER:
        if f"_rnn_{m}_" in name:
            return m
    raise ValueError(f"Cannot parse method from {name}")


def load_chevel_returns(data_root: Path = DATA_ROOT):
    """Return {method: DataFrame[Step, Value, frames_m]} from exported TB CSVs."""
    paths = sorted(data_root.glob(CHE_VEL_RETURN_GLOB))
    if not paths:
        raise FileNotFoundError(f"No CSVs matching {data_root / CHE_VEL_RETURN_GLOB}")
    out = {}
    for path in paths:
        method = parse_chevel_method(path)
        df = pd.read_csv(path)
        df = df.rename(columns={c: c.strip() for c in df.columns})
        df["frames_m"] = df["Step"] / 1e6
        df["method"] = method
        df["source"] = path.name
        out[method] = df
        print(f"{method:5s}  n={len(df):3d}  frames={df['Step'].iloc[-1]/1e6:.2f}M  "
              f"final={df['Value'].iloc[-1]:.1f}  file={path.name}")
    return out


chevel_returns = load_chevel_returns()
display(pd.DataFrame([
    {
        "method": m,
        "n_evals": len(df),
        "frames_M": df["Step"].iloc[-1] / 1e6,
        "best": df["Value"].max(),
        "final": df["Value"].iloc[-1],
        "last_5M_mean": df.loc[df["Step"] >= df["Step"].iloc[-1] - 5e6, "Value"].mean(),
    }
    for m, df in ((m, chevel_returns[m]) for m in METHOD_ORDER if m in chevel_returns)
]).round(2))

In [ ]:
def smooth_series(y, window=11):
    """Centered rolling mean; odd window preferred."""
    y = np.asarray(y, dtype=float)
    if window < 2 or len(y) < window:
        return y
    ker = np.ones(window) / window
    pad = window // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    return np.convolve(yp, ker, mode="valid")


def plot_chevel_returns(returns_dict, ylim=None, smooth=11, title_suffix=""):
    fig, ax = plt.subplots(figsize=(10, 5))
    for method in METHOD_ORDER:
        if method not in returns_dict:
            continue
        df = returns_dict[method]
        x = df["frames_m"].to_numpy()
        y = df["Value"].to_numpy()
        color = METHOD_COLORS[method]
        ax.plot(x, y, color=color, alpha=0.25, linewidth=1)
        ax.plot(
            x, smooth_series(y, smooth),
            color=color, linewidth=2.2, label=method,
        )
    ax.axhline(0, color="0.7", linewidth=0.8, linestyle=":")
    ax.set_xlabel("Frames (M)")
    ax.set_ylabel("Meta-Episode Return (eval)")
    ax.set_title(f"HN Output Scope Experiments")
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()


# Full scale (head crash visible)
plot_chevel_returns(chevel_returns, ylim=None, title_suffix="")

# Clipped view for comparing learning curves without the head explosion
plot_chevel_returns(
    chevel_returns,
    ylim=(-450, 0),
)

## TensorBoard logs (`logs_chevel/`, like the bottom of `exp2.ipynb`)

Same runs as the CSV section above, but read directly from the TensorBoard event files under `logs_chevel/logs_HalfCheetahVel-TaskDist3-v0/chevel_td3_rnn_{full,head,film}_*/` (Step = env frames). Returns are plotted full vs head vs FiLM, followed by the HN weight-divergence scalar `hn_weight_div/mean_pairwise_l2` when present in the logs.

In [ ]:
# Load run_chevel Meta-Episode Return scalars directly from TensorBoard logs
import json
import re

from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

CHEVEL_LOG_ROOT = CODE_DIR / "logs_chevel"
CHEVEL_TD = 3  # run_chevel.sh trains only HalfCheetahVel TaskDist3
RETURN_TAG = "Meta-Episode Return"
HNDIV_TAG = "hn_weight_div/mean_pairwise_l2"


def _chevel_run_dir(log_root: Path, method: str) -> Path:
    runs = sorted(
        (log_root / f"logs_HalfCheetahVel-TaskDist{CHEVEL_TD}-v0").glob(
            f"chevel_td{CHEVEL_TD}_rnn_{method}_*"
        )
    )
    runs = [r for r in runs if r.is_dir()]
    if not runs:
        raise FileNotFoundError(f"No chevel run for method={method} under {log_root}")
    return runs[-1]  # latest if re-run


def load_chevel_tb_returns(log_root: Path = CHEVEL_LOG_ROOT):
    """Return {method: DataFrame[Step, Value, frames_m]} from TB event files."""
    out = {}
    for method in METHOD_ORDER:
        run_dir = _chevel_run_dir(log_root, method)
        ea = EventAccumulator(str(run_dir))
        ea.Reload()
        if RETURN_TAG not in ea.Tags()["scalars"]:
            raise KeyError(f"{RETURN_TAG!r} not in {run_dir}")
        vals = ea.Scalars(RETURN_TAG)
        df = pd.DataFrame([(v.step, v.value) for v in vals], columns=["Step", "Value"])
        df["frames_m"] = df["Step"] / 1e6
        df["method"] = method
        df["source"] = run_dir.name
        out[method] = df
        print(f"{method:5s}  n={len(df):3d}  frames={df['Step'].iloc[-1]/1e6:.2f}M  "
              f"final={df['Value'].iloc[-1]:.1f}  run={run_dir.name}")
    return out


chevel_tb_returns = load_chevel_tb_returns()

_trapz = getattr(np, "trapezoid", np.trapz)
df_chevel_tb_summary = pd.DataFrame([
    {
        "method": m,
        "n_evals": len(df),
        "frames_M": df["Step"].iloc[-1] / 1e6,
        "best": df["Value"].max(),
        "final": df["Value"].iloc[-1],
        "last_10pct_mean": df.loc[df["Step"] >= df["Step"].iloc[-1] * 0.9, "Value"].mean(),
        "auc_mean": _trapz(df["Value"].to_numpy(float), df["Step"].to_numpy(float))
                    / (df["Step"].iloc[-1] - df["Step"].iloc[0]),
    }
    for m, df in ((m, chevel_tb_returns[m]) for m in METHOD_ORDER if m in chevel_tb_returns)
])
print("\nCheetahVel TB summary (Meta-Episode Return):")
display(df_chevel_tb_summary.round(2))

In [ ]:
def plot_chevel_tb_returns(returns_dict, smooth=11, ylim=None):
    """Single TD3 panel; full / head / film (mirrors exp2's plot_grid_returns)."""
    fig, ax = plt.subplots(figsize=(10, 5))
    for method in METHOD_ORDER:
        if method not in returns_dict:
            continue
        df = returns_dict[method]
        x = df["frames_m"].to_numpy()
        y = df["Value"].to_numpy()
        color = METHOD_COLORS[method]
        ax.plot(x, y, color=color, alpha=0.25, linewidth=1)
        ax.plot(x, smooth_series(y, smooth), color=color, linewidth=2.2, label=method)
    ax.axhline(0, color="0.7", linewidth=0.8, linestyle=":")
    ax.set_xlabel("Frames (M)")
    ax.set_ylabel("Meta-Episode Return (eval)")
    ax.set_title(f"CheetahVel TD{CHEVEL_TD} — HN Output Scope Experiments (TensorBoard logs)")
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.legend(frameon=False)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


# Full scale + clipped view (head crash visible in the first)
plot_chevel_tb_returns(chevel_tb_returns, ylim=None)
plot_chevel_tb_returns(chevel_tb_returns, ylim=(-450, 0))

## HN weight divergence (`logs_chevel/`, like the bottom of `exp2.ipynb`)

Scalar `hn_weight_div/mean_pairwise_l2` = mean pairwise L2 between per-task mean **actor** weights generated by the hypernetwork, when logged. Note: the current `logs_chevel` runs predate this logging (no such scalar / `hn_weight_div/*.pt` in those folders, and `run_chevel.sh` does not pass `--log_hn_weight_div`), so the loader below reports that instead of failing; re-running chevel with divergence logging (tracker needs a continuous-task grouping for velocities) would populate it.

In [ ]:
# Load HN weight-divergence from the same TB runs (graceful if not logged)
def load_chevel_hndiv(log_root: Path = CHEVEL_LOG_ROOT):
    """Return {method: DataFrame[Step(iter), Value, frames_m]} (possibly empty)."""
    out = {}
    for method in METHOD_ORDER:
        run_dir = _chevel_run_dir(log_root, method)
        ea = EventAccumulator(str(run_dir))
        ea.Reload()
        if HNDIV_TAG not in ea.Tags()["scalars"]:
            pt_files = sorted((run_dir / "hn_weight_div").glob("iter_*.pt"))
            if not pt_files:
                print(f"{method:5s}: no {HNDIV_TAG!r} in {run_dir.name} (pre-logging run)")
                continue
            # Fallback: saved per-flush dicts {iter, mean_pairwise_l2}
            import torch as _torch
            rows = []
            for p in pt_files:
                d = _torch.load(p, map_location="cpu")
                rows.append((int(d["iter"]), float(d["mean_pairwise_l2"])))
            df = pd.DataFrame(sorted(rows), columns=["Step", "Value"])
        else:
            vals = ea.Scalars(HNDIV_TAG)
            df = pd.DataFrame([(v.step, v.value) for v in vals], columns=["Step", "Value"])
        cfg = json.loads((run_dir / "config.json").read_text())
        frames_per_iter = int(cfg["num_processes"]) * int(cfg["policy_num_steps"])
        df["frames_m"] = df["Step"] * frames_per_iter / 1e6
        df["method"] = method
        df["source"] = run_dir.name
        out[method] = df
        print(f"{method:5s}  n={len(df):2d}  final={df['Value'].iloc[-1]:.2f}  run={run_dir.name}")
    return out


chevel_hndiv = load_chevel_hndiv()

if chevel_hndiv:
    display(pd.DataFrame([
        {"method": m, "n_logs": len(df), "final": df["Value"].iloc[-1],
         "max": df["Value"].max(), "mean": df["Value"].mean()}
        for m, df in chevel_hndiv.items()
    ]).round(2))

    fig, ax = plt.subplots(figsize=(10, 5))
    for method in METHOD_ORDER:
        if method not in chevel_hndiv:
            continue
        df = chevel_hndiv[method]
        ax.plot(df["frames_m"], df["Value"], "o-", color=METHOD_COLORS[method],
                linewidth=2, markersize=4, label=method)
    ax.set_xlabel("Frames (M, approx: iter x frames-per-iter)")
    ax.set_ylabel("HN weight div: mean pairwise L2")
    ax.set_title(f"CheetahVel TD{CHEVEL_TD} — HN Weight Divergence")
    ax.legend(frameon=False)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()
else:
    print("HN weight divergence not available for these chevel logs; see note above.") 